In [1]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

!date

Fri Jul 26 16:29:56 PDT 2024


# HCES Data Extraction
We have HCES data to inform rice consumption (of PSD-distributed and non-PSD rice) by women and birthing people of 
reproductive age (WBPRA) and U5 children in India. We need to do further investigation as to what percentage of PSD-
distributed rice has been fortified with iron and folate.

HCES data and documentation is saved here: 
https://uwnetid.sharepoint.com/:f:/r/sites/ihme_simulation_science_team/Shared%20Documents/Research/LSFF/07_Data/HCES_22_data_files?csf=1&web=1&e=wmxQSD

Item codes for 'Rice' include: 101, 102, and 061. See Section 5.1 of the above documentation for more context.
- Item code 101 signifies rice procured through PDS, using ration card.
- Item code 061 signifies rice procured through PDS, free of charge.
- Item code 102 signifies rice procured/consumed from other sources. (NOTE: Presumably this is unfortified rice.)


We also will need to approximate DHS wealth quintiles in the HCES data, by using similar variables as is used by the DHS to 
calculate wealth quintiles in India (see DHS wealth quintile documentation here: 
https://dhsprogram.com/programming/wealth%20index/India%20DHS%202015-16/India%202015-16%20sps.txt)

In this notebook, we extract the raw data from the HCES website and do some initial processing in order to create files 
that we can further tabulate based on our project needs in a later step. For now, we process these raw data into dataframes 
that have a row for each household/individual with the following columns:
- Whether or not rice was consumed (binary variable - this will be used to calculate coverage percentages later)
- Amount of PDS rice consumed (grams per day - currently in raw HCES, *I think* this is in kg per 2 weeks; this will be used to calculate fortified rice consumption amount) 
- Amount of non-PDS rice consumed (grams per day - currently in raw HCES, *I think* this is in kg per 2 weeks; this will be used to calculate unfortified rice consumption amount) 
- Wealth composite score/quintile 
- Sex/gender (will be used to calculate WBPRA)
- Age (will be used to tabulate age groups)

In [109]:
nonpds_df = df2[df2[3] == 102]
nonpds_df
# non-PDS rice

,0,1,2,3,4,5,6,7,8,9
0,HCES2022655561010131113011 101202 201,F,5,102,NaN,NaN,40,1200.0,1.0,35560.0
26,HCES2022655561010131113011 101202 301,F,5,102,40,950,40,950.0,2.0,30331.0
53,HCES2022655561010131113011 101202 302,F,5,102,30,900,36,1020.0,3.0,30331.0
80,HCES2022655561010131113011 101202 303,F,5,102,NaN,NaN,25,500.0,1.0,30331.0
106,HCES2022655561010131113011 101202 304,F,5,102,20,600,20,600.0,2.0,30331.0
...,...,...,...,...,...,...,...,...,...,...
9314415,HCES20223951022828313220210228101 314,F,5,102,NaN,NaN,15.00,750.0,1.0,73515.0
9314452,HCES20223951022828313220210228101 315,F,5,102,NaN,NaN,25.00,1300.0,1.0,73515.0
9314493,HCES20223951022828313220210228101 316,F,5,102,NaN,NaN,25.00,1300.0,1.0,73515.0
9314532,HCES20223951022828313220210228101 317,F,5,102,NaN,NaN,25.00,1300.0,1.0,73515.0


In [103]:
pds_df = df2[(df2[3] == 101) | (df2[3] == 61)]
pds_df
# PDS rice

,0,1,2,3,4,5,6,7,8,9
133,HCES2022655561010131113011 101202 305,F,5,101,NaN,NaN,20,240.0,1.0,30331.0
354,HCES2022655561010131113011 101202 314,F,5,101,NaN,NaN,10,130.0,1.0,30331.0
448,HCES2022655531010131213011 201212 301,F,5,101,NaN,NaN,50.00,250.0,1.0,23213.0
465,HCES2022655531010131213011 201212 302,F,5,101,NaN,NaN,50.000,250.0,1.0,23213.0
489,HCES2022655531010131213011 201212 303,F,5,101,NaN,NaN,40.000,120.0,1.0,23213.0
...,...,...,...,...,...,...,...,...,...,...
9314299,HCES20223951022828313220210228101 311,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314340,HCES20223951022828313220210228101 312,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314414,HCES20223951022828313220210228101 314,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314531,HCES20223951022828313220210228101 317,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0


In [104]:
# Columns 6 and 7 denote the quantity of total consumption (in kg), with column 6 being the integer and column 7 
# the fractional part? (See section 3.3.3.3 in HCES Volume I documentation) - So should I concatenate these two columns
# together? 
pds_df[6].describe()

count     179216
unique      1357
top           20
freq       14994
Name: 6, dtype: object

In [57]:
pds_df[7].value_counts().sort_values()
# All of these values should be max 3 digits - that one 4 digit value is probably an error, I think it's fine

928.0        1
470.0        1
408.0        1
1620.0       1
193.0        1
          ... 
10.0      3766
60.0      4060
20.0      4655
15.0      4765
30.0      5194
Name: 7, Length: 485, dtype: int64

In [110]:
# Let's rename the variables in these dfs so they are easier to deal with! 
pds_df = pds_df.rename({0: 'common_id', 6: 'pds_total_consumption_int', 7: 'pds_total_consumption_decimal'}, axis=1)
nonpds_df = nonpds_df.rename({0: 'common_id', 6: 'nonpds_total_consumption_int', 7: 'nonpds_total_consumption_decimal'}, axis=1)

In [106]:
pds_df

,common_id,1,2,3,4,5,pds_total_consumption_int,pds_total_consumption_decimal,8,9
133,HCES2022655561010131113011 101202 305,F,5,101,NaN,NaN,20,240.0,1.0,30331.0
354,HCES2022655561010131113011 101202 314,F,5,101,NaN,NaN,10,130.0,1.0,30331.0
448,HCES2022655531010131213011 201212 301,F,5,101,NaN,NaN,50.00,250.0,1.0,23213.0
465,HCES2022655531010131213011 201212 302,F,5,101,NaN,NaN,50.000,250.0,1.0,23213.0
489,HCES2022655531010131213011 201212 303,F,5,101,NaN,NaN,40.000,120.0,1.0,23213.0
...,...,...,...,...,...,...,...,...,...,...
9314299,HCES20223951022828313220210228101 311,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314340,HCES20223951022828313220210228101 312,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314414,HCES20223951022828313220210228101 314,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314531,HCES20223951022828313220210228101 317,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0


Now we need to use these dfs to find rice consumption amounts in our populations of interest: WBPRA and U5 children. We 
will look to household characteristics to determine these numbers.

Info we have: 
- Item 5.5: household size (Level 03, Question 2.1) 

In [58]:
dfh = pd.read_fwf('data/hces22_lvl_03.TXT', header=None, widths = [38,1,2,2,1,3,5,1,1,1,1,1,1,1,1,1,9,1,1,1,1,1,2,1,2,3,1,2,1,1,1,1,2,15])
dfh

,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,HCES2022655561010131113011 101202 201,H,3,5,1,332.0,68200.0,1.0,2.0,NaN,...,2,0.0,1,2.0,3,2.0,2,2,0,35560.0
1,HCES2022655561010131113011 101202 301,H,3,6,1,931.0,42909.0,3.0,NaN,NaN,...,2,0.0,1,2.0,2,2.0,2,2,0,30331.0
2,HCES2022655561010131113011 101202 302,H,3,8,1,833.0,49211.0,1.0,2.0,NaN,...,2,0.0,1,2.0,3,2.0,2,2,0,30331.0
3,HCES2022655561010131113011 101202 303,H,3,4,1,142.0,47713.0,1.0,2.0,NaN,...,2,0.0,1,2.0,0,2.0,2,2,0,30331.0
4,HCES2022655561010131113011 101202 304,H,3,4,1,833.0,49211.0,1.0,2.0,NaN,...,2,0.0,1,2.0,3,2.0,2,2,0,30331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261741,HCES20223951022828313220210228101 314,H,3,2,1,832.0,49219.0,1.0,NaN,NaN,...,11,10.0,1,1.0,4,NaN,1,2,0,73515.0
261742,HCES20223951022828313220210228101 315,H,3,4,1,216.0,85500.0,2.0,NaN,NaN,...,2,0.0,1,1.0,0,NaN,1,2,0,73515.0
261743,HCES20223951022828313220210228101 316,H,3,3,1,411.0,84119.0,2.0,NaN,NaN,...,2,0.0,1,1.0,4,NaN,1,2,0,73515.0
261744,HCES20223951022828313220210228101 317,H,3,5,1,112.0,47711.0,1.0,NaN,NaN,...,2,0.0,1,1.0,4,NaN,1,2,0,73515.0


In [60]:
dfh[3].value_counts()
# This should be household size! The fact that there are a couple very large households (e.g. 20+ members) is a little 
# surprising - maybe we can investigate to confirm these are some kind of GQ. 

# Although not sure we need household size, pausing this for now.

4     64290
5     46960
3     42596
2     31243
6     27890
1     18181
7     13421
8      7183
9      4008
10     2573
11     1390
12      790
13      446
14      266
15      164
16      138
17       68
18       44
20       30
19       25
21       13
22        9
24        5
27        3
28        2
23        2
25        2
30        1
37        1
31        1
29        1
Name: 3, dtype: int64

Let's see what we have for household member characteristics (Section 3: Details of the household members): 
- Column 4: gender (1 - male, 2 - female, 3 - transgender (hijras, eunuchs)) 
- Column 5: age (years)

This information is saved in the Level 02 file.

In [65]:
dfh = pd.read_fwf('data/hces22_lvl_02.TXT', header=None, widths = [38,1,2,2,1,1,3,1,2,2,1,2,1,2,2,2,2,2,2,1,1,15])
dfh

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,HCES2022655561010131113011 101202 201,H,2,1,1,1,48,2,6,12.0,...,2.0,0.0,0.0,0.0,0.0,58.0,11,3,5,560.0
1,HCES2022655561010131113011 101202 201,H,2,2,2,2,46,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,5,560.0
2,HCES2022655561010131113011 101202 201,H,2,3,5,1,24,1,13,18.0,...,2.0,NaN,NaN,NaN,NaN,58.0,11,3,5,560.0
3,HCES2022655561010131113011 101202 201,H,2,4,5,1,18,1,7,13.0,...,2.0,NaN,NaN,NaN,NaN,56.0,11,3,5,560.0
4,HCES2022655561010131113011 101202 201,H,2,5,5,2,21,1,12,17.0,...,2.0,NaN,NaN,NaN,NaN,54.0,11,3,5,560.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127034,HCES20223492023737102120110201201 313,H,2,1,1,1,40,2,12,17.0,...,2.0,0.0,0.0,0.0,5.0,55.0,11,9,3,8.0
1127035,HCES20223492023737102120110201201 313,H,2,2,2,2,38,2,6,12.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127036,HCES20223492023737102120110201201 313,H,2,3,5,1,12,1,4,7.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127037,HCES20223492023737102120110201201 313,H,2,4,5,2,7,1,3,2.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0


In [68]:
dfh[5].value_counts()
# Gender 

1    574615
2    552033
3       391
Name: 5, dtype: int64

In [69]:
dfh[6].value_counts()
# Age
# WHO considers women 15-49 years old to be of reproductive age

30     32144
35     31767
40     31372
45     30790
25     26620
       ...  
108        4
115        2
120        1
117        1
106        1
Name: 6, Length: 113, dtype: int64

In [87]:
wra_df = dfh[(dfh[5] == 2) & ((dfh[6] >= 15) & (dfh[6] <= 49))] 
wra_df

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
1,HCES2022655561010131113011 101202 201,H,2,2,2,2,46,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,5,560.0
4,HCES2022655561010131113011 101202 201,H,2,5,5,2,21,1,12,17.0,...,2.0,NaN,NaN,NaN,NaN,54.0,11,3,5,560.0
6,HCES2022655561010131113011 101202 301,H,2,2,2,2,45,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
9,HCES2022655561010131113011 101202 301,H,2,5,5,2,19,1,7,13.0,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
13,HCES2022655561010131113011 101202 302,H,2,3,4,2,33,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,58.0,11,3,0,331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127006,HCES20223492023737102120110201201 307,H,2,2,2,2,37,2,7,14.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127013,HCES20223492023737102120110201201 308,H,2,5,5,2,28,1,13,17.0,...,2.0,0.0,0.0,0.0,2.0,58.0,11,9,3,8.0
1127022,HCES20223492023737102120110201201 310,H,2,3,4,2,28,2,7,14.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127031,HCES20223492023737102120110201201 312,H,2,2,2,2,45,2,6,12.0,...,2.0,0.0,0.0,0.0,3.0,57.0,11,9,3,8.0


In [88]:
# Let's rename the columns so they are easier to deal with! 
wra_df = wra_df.rename({0: 'common_id', 5: 'gender', 6: 'age'}, axis=1)
wra_df

,common_id,1,2,3,4,gender,age,7,8,9,...,12,13,14,15,16,17,18,19,20,21
1,HCES2022655561010131113011 101202 201,H,2,2,2,2,46,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,5,560.0
4,HCES2022655561010131113011 101202 201,H,2,5,5,2,21,1,12,17.0,...,2.0,NaN,NaN,NaN,NaN,54.0,11,3,5,560.0
6,HCES2022655561010131113011 101202 301,H,2,2,2,2,45,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
9,HCES2022655561010131113011 101202 301,H,2,5,5,2,19,1,7,13.0,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
13,HCES2022655561010131113011 101202 302,H,2,3,4,2,33,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,58.0,11,3,0,331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127006,HCES20223492023737102120110201201 307,H,2,2,2,2,37,2,7,14.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127013,HCES20223492023737102120110201201 308,H,2,5,5,2,28,1,13,17.0,...,2.0,0.0,0.0,0.0,2.0,58.0,11,9,3,8.0
1127022,HCES20223492023737102120110201201 310,H,2,3,4,2,28,2,7,14.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127031,HCES20223492023737102120110201201 312,H,2,2,2,2,45,2,6,12.0,...,2.0,0.0,0.0,0.0,3.0,57.0,11,9,3,8.0


In [89]:
wra_df.age.value_counts().sort_values(ascending=False)

30    15849
40    14885
35    14779
45    14364
25    13135
28    12756
32    12130
18    11988
22    11978
20    11608
38    11528
15    10250
16     9918
42     9829
26     9819
24     9576
23     9276
17     9129
48     8496
19     8487
21     8119
27     8112
36     7542
34     6255
33     6176
29     6042
37     5530
46     5093
43     5028
39     4684
47     4510
31     4430
44     3963
41     3386
49     3348
Name: age, dtype: int64

In [94]:
u5_df = dfh[dfh[6] < 5] 
u5_df

# TODO: double-check that U5 children is not inclusive to 5 year olds

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
18,HCES2022655561010131113011 101202 302,H,2,8,6,1,3,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
22,HCES2022655561010131113011 101202 303,H,2,4,5,1,4,1,3,1.0,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
26,HCES2022655561010131113011 101202 304,H,2,4,6,2,4,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
36,HCES2022655561010131113011 101202 306,H,2,5,6,1,4,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
50,HCES2022655561010131113011 101202 309,H,2,4,5,2,3,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1126897,HCES20223492723737101120110101201 104,H,2,5,6,1,2,1,1,NaN,...,3.0,0.0,NaN,NaN,NaN,90.0,11,3,9,54.0
1126903,HCES20223492723737101120110101201 105,H,2,4,6,1,1,1,1,NaN,...,3.0,NaN,NaN,NaN,NaN,90.0,11,3,9,54.0
1126942,HCES20223492723737101120110101201 302,H,2,3,5,1,4,1,1,NaN,...,3.0,24.0,0.0,NaN,NaN,66.0,11,2,3,85.0
1126969,HCES20223492023737102120110201201 203,H,2,6,6,1,2,1,1,NaN,...,2.0,0.0,0.0,0.0,0.0,60.0,11,8,4,8.0


In [95]:
u5_df = u5_df.rename({0: 'common_id', 5: 'gender', 6: 'age'}, axis=1)
u5_df

,common_id,1,2,3,4,gender,age,7,8,9,...,12,13,14,15,16,17,18,19,20,21
18,HCES2022655561010131113011 101202 302,H,2,8,6,1,3,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
22,HCES2022655561010131113011 101202 303,H,2,4,5,1,4,1,3,1.0,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
26,HCES2022655561010131113011 101202 304,H,2,4,6,2,4,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
36,HCES2022655561010131113011 101202 306,H,2,5,6,1,4,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
50,HCES2022655561010131113011 101202 309,H,2,4,5,2,3,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1126897,HCES20223492723737101120110101201 104,H,2,5,6,1,2,1,1,NaN,...,3.0,0.0,NaN,NaN,NaN,90.0,11,3,9,54.0
1126903,HCES20223492723737101120110101201 105,H,2,4,6,1,1,1,1,NaN,...,3.0,NaN,NaN,NaN,NaN,90.0,11,3,9,54.0
1126942,HCES20223492723737101120110101201 302,H,2,3,5,1,4,1,1,NaN,...,3.0,24.0,0.0,NaN,NaN,66.0,11,2,3,85.0
1126969,HCES20223492023737102120110201201 203,H,2,6,6,1,2,1,1,NaN,...,2.0,0.0,0.0,0.0,0.0,60.0,11,8,4,8.0


Now, let's merge together wra_df, pds_df, and nonpds_df, and separately u5_df, pds_df, and nonpds_df so we can actually
calculate consumption amounts of PDS and non-PDS rice! 

In [111]:
wra_rice_df = pd.merge(wra_df, pds_df, on='common_id', how='inner')
wra_rice_df = pd.merge(wra_rice_df, nonpds_df, on='common_id', how='inner')
wra_rice_df

,common_id,1_x,2_x,3_x,4_x,gender,age,7,8_x,9_x,...,9_y,1,2,3,4,5_y,nonpds_total_consumption_int,nonpds_total_consumption_decimal,8,9
0,HCES2022655561010131113011 101202 305,H,2,2,2,2,46,2,1,NaN,...,30331.0,F,5,102,10,300,10,300.0,2.0,30331.0
1,HCES2022655561010131113011 101202 305,H,2,5,5,2,24,1,7,13.0,...,30331.0,F,5,102,10,300,10,300.0,2.0,30331.0
2,HCES2022655561010131113011 101202 314,H,2,2,2,2,44,2,1,NaN,...,30331.0,F,5,102,30,900,30,900.0,2.0,30331.0
3,HCES2022655531010131213011 201212 302,H,2,4,4,2,33,2,3,4.0,...,23213.0,F,5,102,NaN,NaN,40.000,800.0,1.0,23213.0
4,HCES2022655531010131213011 201212 302,H,2,9,5,2,23,1,5,8.0,...,23213.0,F,5,102,NaN,NaN,40.000,800.0,1.0,23213.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151038,HCES20223492023737102120110201201 307,H,2,2,2,2,37,2,7,14.0,...,938.0,F,5,102,NaN,NaN,10,400.0,1.0,938.0
151039,HCES20223492023737102120110201201 308,H,2,5,5,2,28,1,13,17.0,...,938.0,F,5,102,NaN,NaN,10,400.0,1.0,938.0
151040,HCES20223492023737102120110201201 310,H,2,3,4,2,28,2,7,14.0,...,938.0,F,5,102,NaN,NaN,10,400.0,1.0,938.0
151041,HCES20223492023737102120110201201 312,H,2,2,2,2,45,2,6,12.0,...,938.0,F,5,102,NaN,NaN,10,400.0,1.0,938.0


In [112]:
wra_rice_df.columns

Index([                       'common_id',                              '1_x',
                                    '2_x',                              '3_x',
                                    '4_x',                           'gender',
                                    'age',                                  7,
                                    '8_x',                              '9_x',
                                       10,                                 11,
                                       12,                                 13,
                                       14,                                 15,
                                       16,                                 17,
                                       18,                                 19,
                                       20,                                 21,
                                    '1_y',                              '2_y',
                                    '3_y',          

In [114]:
wra_rice_df['pds_total_consumption_int'] = wra_rice_df['pds_total_consumption_int'].astype(str)
wra_rice_df['nonpds_total_consumption_int'] = wra_rice_df['nonpds_total_consumption_int'].astype(str)
wra_rice_df['pds_total_consumption_decimal'] = wra_rice_df['pds_total_consumption_decimal'].astype(str)
wra_rice_df['nonpds_total_consumption_decimal'] = wra_rice_df['nonpds_total_consumption_decimal'].astype(str)

wra_rice_df['pds_total_consumption_decimal'] = wra_rice_df.pds_total_consumption_decimal.str.replace('.0', '')
wra_rice_df['nonpds_total_consumption_decimal'] = wra_rice_df.nonpds_total_consumption_decimal.str.replace('.0', '')

wra_rice_df['pds_total_consumption'] = wra_rice_df['pds_total_consumption_decimal'] + '.' + wra_rice_df['pds_total_consumption_decimal']
wra_rice_df['nonpds_total_consumption'] = wra_rice_df['nonpds_total_consumption_decimal'] + '.' + wra_rice_df['nonpds_total_consumption_decimal']

wra_rice_df['pds_total_consumption'] = pd.to_numeric(wra_rice_df['pds_total_consumption'], errors='coerce')
wra_rice_df['nonpds_total_consumption'] = pd.to_numeric(wra_rice_df['nonpds_total_consumption'], errors='coerce')

/tmp/ipykernel_2352966/1422480942.py:6: FutureWarning: The default value of regex will change from True to False in a future version.
  wra_rice_df['pds_total_consumption_decimal'] = wra_rice_df.pds_total_consumption_decimal.str.replace('.0', '')
/tmp/ipykernel_2352966/1422480942.py:7: FutureWarning: The default value of regex will change from True to False in a future version.
  wra_rice_df['nonpds_total_consumption_decimal'] = wra_rice_df.nonpds_total_consumption_decimal.str.replace('.0', '')


In [123]:
wra_rice_df.pds_total_consumption.describe()

count    43316.000000
mean        46.363408
std         90.318725
min          0.000000
25%          5.500000
50%         24.240000
75%         45.450000
max       2625.262500
Name: pds_total_consumption, dtype: float64

In [124]:
wra_rice_df.nonpds_total_consumption.describe()

count    132320.000000
mean        111.013397
std         294.423469
min           0.000000
25%           0.000000
50%           3.300000
75%          12.120000
max        4185.418500
Name: nonpds_total_consumption, dtype: float64

In [118]:
u5_rice_df = pd.merge(u5_df, pds_df, on='common_id', how='inner')
u5_rice_df = pd.merge(u5_rice_df, nonpds_df, on='common_id', how='inner')
u5_rice_df

,common_id,1_x,2_x,3_x,4_x,gender,age,7,8_x,9_x,...,9_y,1,2,3,4,5_y,nonpds_total_consumption_int,nonpds_total_consumption_decimal,8,9
0,HCES2022655531010131213011 201212 302,H,2,5,6,1,2,1,1,NaN,...,23213.0,F,5,102,NaN,NaN,40.000,800.0,1.0,23213.0
1,HCES2022655531010131213011 201212 302,H,2,6,6,2,4,1,1,NaN,...,23213.0,F,5,102,NaN,NaN,40.000,800.0,1.0,23213.0
2,HCES2022655531010131213011 201212 309,H,2,5,6,1,4,1,3,3.0,...,23213.0,F,5,102,NaN,NaN,10.00,200.0,1.0,23213.0
3,HCES2022655531010131213011 201212 310,H,2,9,6,1,3,1,1,NaN,...,23213.0,F,5,102,NaN,NaN,10.000,200.0,1.0,23213.0
4,HCES2022655531010131213011 201212 311,H,2,4,6,1,0,1,1,NaN,...,23213.0,F,5,102,NaN,NaN,20,400.0,1.0,23213.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35305,HCES2022349282373710112019 201201 207,H,2,6,6,1,4,1,1,NaN,...,1272.0,F,5,102,NaN,NaN,25,800.0,1.0,1272.0
35306,HCES20223492723737101120110101201 105,H,2,4,6,1,1,1,1,NaN,...,3954.0,F,5,102,NaN,NaN,10,320.0,1.0,3954.0
35307,HCES20223492723737101120110101201 105,H,2,4,6,1,1,1,1,NaN,...,3954.0,F,5,102,NaN,NaN,10,320.0,1.0,3954.0
35308,HCES20223492023737102120110201201 203,H,2,6,6,1,2,1,1,NaN,...,848.0,F,5,102,NaN,NaN,10,400.0,1.0,848.0


In [119]:
u5_rice_df['pds_total_consumption_int'] = u5_rice_df['pds_total_consumption_int'].astype(str)
u5_rice_df['nonpds_total_consumption_int'] = u5_rice_df['nonpds_total_consumption_int'].astype(str)
u5_rice_df['pds_total_consumption_decimal'] = u5_rice_df['pds_total_consumption_decimal'].astype(str)
u5_rice_df['nonpds_total_consumption_decimal'] = u5_rice_df['nonpds_total_consumption_decimal'].astype(str)

u5_rice_df['pds_total_consumption_decimal'] = u5_rice_df.pds_total_consumption_decimal.str.replace('.0', '')
u5_rice_df['nonpds_total_consumption_decimal'] = u5_rice_df.nonpds_total_consumption_decimal.str.replace('.0', '')

u5_rice_df['pds_total_consumption'] = u5_rice_df['pds_total_consumption_decimal'] + '.' + u5_rice_df['pds_total_consumption_decimal']
u5_rice_df['nonpds_total_consumption'] = u5_rice_df['nonpds_total_consumption_decimal'] + '.' + u5_rice_df['nonpds_total_consumption_decimal']

u5_rice_df['pds_total_consumption'] = pd.to_numeric(u5_rice_df['pds_total_consumption'], errors='coerce')
u5_rice_df['nonpds_total_consumption'] = pd.to_numeric(u5_rice_df['nonpds_total_consumption'], errors='coerce')

/tmp/ipykernel_2352966/3869122219.py:6: FutureWarning: The default value of regex will change from True to False in a future version.
  u5_rice_df['pds_total_consumption_decimal'] = u5_rice_df.pds_total_consumption_decimal.str.replace('.0', '')
/tmp/ipykernel_2352966/3869122219.py:7: FutureWarning: The default value of regex will change from True to False in a future version.
  u5_rice_df['nonpds_total_consumption_decimal'] = u5_rice_df.nonpds_total_consumption_decimal.str.replace('.0', '')


In [121]:
u5_rice_df.pds_total_consumption.describe()

count    10116.000000
mean        44.510662
std         83.211363
min          0.000000
25%          5.500000
50%         24.240000
75%         45.450000
max       1875.187500
Name: pds_total_consumption, dtype: float64

In [122]:
u5_rice_df.nonpds_total_consumption.describe()

count    33114.000000
mean       102.981819
std        285.691983
min          0.000000
25%          0.000000
50%          4.400000
75%         10.100000
max       3395.339500
Name: nonpds_total_consumption, dtype: float64

# Calculate wealth index

In order to add wealth stratifications into our HCES data, we will emulate DHS methods for calculating wealth quintiles.
To do this, we will compare the variables used to calculate wealth index in the latest DHS survey in India () with the 
variables available to us in HCES. 

DHS wealth index variables include: 
- Source of drinking water
- Type of toilet facility
- Electricity
- Mattress
- Pressure cooker
- Chair
- Cot or bed
- Table
- Electric fan
- Radio or transistor
- Black and white television
- Colour television
- Sewing machine
- Mobile telephone
- Telephone (non-mobile)
- Internet
- Computer
- Refrigerator
- Air conditioner/cooler
- Washing machine
- Watch or clock
- Bicycle
- Motorcycle or Scooter
- Animal-drawn cart
- Car
- Water pump
- Thresher
- Tractor
- Type of cooking fuel
- Main material of floor
- Main roof material
- Main wall material
- Hectares for agricultural land
- Out of this land, how much is irrigated?
- Cows / bulls / buffaloes
- Camels
- Horses / donkeys / mules
- Goats
- Sheep
- Chickens / ducks
- Bank account
- Members per sleeping room
- Compute urban and rural variables coded (1/0) for filters later
- Toilet facility by shared/not shared
- Land area by units - if there are separate units - need to convert them to one unit

HCES variables:
- Type of land owned
- What is the total area of all owned (owned and possessed or leased out) land (within the country) by the household as on the date of survey (area in acre)?(upto two places of decimal) 
- Basic building Material used for major portion of the wall of the dwelling Unit
- Basic building Material used for construction of the major portion of the outer exposed part of the roof of the dwelling unit
- Basic Building Material used for construction of the major portion of the floor of the dwelling Unit
- Source of Drinking Water (Last 365 days)
- Type of latrine in which the household has access
- Primary source of energy of the household for cooking
- Household has internet facility as on the date of the survey
- Whether household possessed one or more item as on the date of the survey- Television 
- Whether household possessed one or more item as on the date of the survey- Radio
- Whether household possessed one or more item as on the date of the survey - Laptop/PC
- Whether household possessed one or more item as on the date of the survey- Mobile handset
- Whether household possessed one or more item as on the date of the survey- Bicycle
- Whether household possessed one or more item as on the date of the survey- Motorcycle, scooter 
- Whether household possessed one or more item as on the date of the survey- Motor car/jeep/van
- Whether household possessed one or more item as on the date of the survey- Trucks
- Whether household possessed one or more item as on the date of the survey - Animal cart
- Whether household possessed one or more item as on the date of the survey- Refrigerator
- Whether household possessed one or more item as on the date of the survey- Washing machine
- Whether household possessed one or more item as on the date of the survey- Air conditioner/air cooler 
- Type of multichannel television facility is used by the household as on the date of the survey

Validation method: Can double-check methods by replicating DHS wealth calculation process with the India DHS data? And test with only the variables that we also have in HCES, see how similar the score is, once we slice up into 5 parts.
(Compare number of people in wealth quintile 1 in DHS to number in HCES)

Crosswalk spreadsheet saved here: https://uwnetid.sharepoint.com/sites/ihme_simulation_science_team/_layouts/15/doc.aspx?sourcedoc={eddcbf51-3462-47d7-b847-79edea5046c6}&action=edit

In [2]:
dhs_wealth_df = pd.read_excel('data/dhs_wealth_calculation/India-DHS-wealth-variables.xlsx').rename(columns={'Variable':'dhs_name', 'If has': 'if_has', 'If does not have':'if_does_not_have'})
dhs_wealth_df

,dhs_name,if_has,if_does_not_have
0,QH25_11 Source of drinking water: Piped - into...,0.101940,-0.020890
1,QH25_12 Source of drinking water: Piped - into...,0.036502,-0.005556
2,QH25_13 Source of drinking water: Piped - publ...,-0.011698,0.001952
3,QH25_21 Source of drinking water: Tube well / ...,-0.048200,0.029975
4,QH25_31 Source of drinking water: Dug well - p...,0.042509,-0.001504
...,...,...,...
127,QH53 Bank account,0.009841,-0.080892
128,DOMESTHH Domestic staff listed in HH,0.125818,-0.000371
129,HOUSE Owns a house,-0.001762,0.008517
130,LAND Owns land,-0.015278,0.014861


In [3]:
# Load DHS-HCES wealth variable crosswalk that I made in Excel 
crosswalk_df = pd.read_excel('data/dhs_wealth_calculation/dhs_hces_wealth_variable_crosswalk.xlsx')
crosswalk_df

,dhs_name,hces_name,hces_code,other_hces_name,other_hces_code
0,QH25_11 Source of drinking water: Piped - into...,drinking_water_source_2,2.0,NaN,NaN
1,QH25_12 Source of drinking water: Piped - into...,drinking_water_source_3,3.0,NaN,NaN
2,QH25_13 Source of drinking water: Piped - publ...,drinking_water_source_5,5.0,NaN,NaN
3,QH25_21 Source of drinking water: Tube well / ...,drinking_water_source_6,6.0,NaN,NaN
4,QH25_31 Source of drinking water: Dug well - p...,drinking_water_source_8,8.0,NaN,NaN
...,...,...,...,...,...
137,QH53 Bank account,NaN,NaN,NaN,NaN
138,DOMESTHH Domestic staff listed in HH,NaN,NaN,NaN,NaN
139,HOUSE Owns a house,NaN,NaN,NaN,NaN
140,LAND Owns land,land_ownership,NaN,NaN,NaN


In [4]:
# Merge crosswalk and DHS wealth variable weights so that I can calculate wealth composite score for each household
wealth_scores_df = crosswalk_df.merge(dhs_wealth_df, on = 'dhs_name')
# Drop the secondary HCES codes for now because of complexity; can come back and add in once we've done a preliminary
# calculation first.
wealth_scores_df = wealth_scores_df.drop(columns = ['hces_code','other_hces_name','other_hces_code'])
wealth_scores_df

,dhs_name,hces_name,if_has,if_does_not_have
0,QH25_11 Source of drinking water: Piped - into...,drinking_water_source_2,0.101940,-0.020890
1,QH25_12 Source of drinking water: Piped - into...,drinking_water_source_3,0.036502,-0.005556
2,QH25_13 Source of drinking water: Piped - publ...,drinking_water_source_5,-0.011698,0.001952
3,QH25_21 Source of drinking water: Tube well / ...,drinking_water_source_6,-0.048200,0.029975
4,QH25_31 Source of drinking water: Dug well - p...,drinking_water_source_8,0.042509,-0.001504
...,...,...,...,...
136,QH53 Bank account,NaN,0.009841,-0.080892
137,DOMESTHH Domestic staff listed in HH,NaN,0.125818,-0.000371
138,HOUSE Owns a house,NaN,-0.001762,0.008517
139,LAND Owns land,land_ownership,-0.015278,0.014861


In [5]:
# Check for duplicated 'hces_name' entries
duplicates = wealth_scores_df[wealth_scores_df.duplicated('hces_name', keep=False)]
print(duplicates)

                                              dhs_name  \
11   QH25_71 Source of drinking water: Cart with sm...   
15   QH25_92 Source of drinking water: Community RO...   
16             QH25_96 Source of drinking water: Other   
19   QH31_13 Type of toilet facility: Flush - to pi...   
20   QH31_14 Type of toilet facility: Flush - to so...   
..                                                 ...   
135                  QH44_96 Main wall material: Other   
136                                  QH53 Bank account   
137               DOMESTHH Domestic staff listed in HH   
138                                 HOUSE Owns a house   
140       memsleep Number of members per sleeping room   

                    hces_name    if_has  if_does_not_have  
11   drinking_water_source_19  0.035055         -0.000066  
15                        NaN  0.068996         -0.000369  
16   drinking_water_source_19  0.005744         -0.000011  
19             latrine_type_5  0.026690         -0.002531  
20 

In [6]:
wealth_scores_df = wealth_scores_df.groupby('hces_name').agg({
    'if_has': 'mean',
    'if_does_not_have': 'mean'
}).reset_index()

# For now I'm going to deal with these duplicate values by averaging the wealth scores for different categories together...
# can change later if this is not valid

In [7]:
# Now that I have a crosswalk df I will load the raw HCES files that have the wealth variables I need and merge them all
# into the same dataframe.

In [8]:
df3 = pd.read_fwf('data/raw_hces/hces22_lvl_03.TXT', 
                  header = None, 
                  widths = [38,1,2,2,1,3,5,1,1,1,1,1,1,1,1,1,9,1,1,1,1,1,2,1,2,3,1,2,1,1,1,1,2,15], 
                  names=['common_id','q_num','level','household_size','econ_acts','nco_code','nic_code',
                        'max_income_acts','maj_income_self_empl_agri','maj_income_wage_agri','maj_income_casual_agri',
                        'household_type','religion_hoh','social_group_hoh','land_ownership','land_ownership_type',
                        'land_ownership_area','dwelling_unit','dwelling_unit_type','building_material_wall',
                        'building_material_roof','building_material_floor','energy_cooking','energy_lighting',
                        'drinking_water_source','time_to_water_source','latrine_access_type','latrine_type',
                        'ration_card_type','locality_rent','benefit_from_pmgky','0_18_deaths','num_of_0_18_deaths',
                        'multiplier']
                 )
# Only keep the data that I need for the wealth calculation
df3 = df3[['common_id','land_ownership','latrine_access_type','latrine_type','building_material_wall','drinking_water_source',
                        'building_material_roof','building_material_floor','energy_cooking','multiplier']]
df3

,common_id,land_ownership,latrine_access_type,latrine_type,building_material_wall,drinking_water_source,building_material_roof,building_material_floor,energy_cooking,multiplier
0,HCES2022655561010131113011 101202 201,1,1,2.0,6.0,2,8.0,8.0,2,35560.0
1,HCES2022655561010131113011 101202 301,1,1,2.0,6.0,2,7.0,8.0,2,30331.0
2,HCES2022655561010131113011 101202 302,1,1,2.0,6.0,2,7.0,8.0,2,30331.0
3,HCES2022655561010131113011 101202 303,1,1,2.0,6.0,2,7.0,8.0,2,30331.0
4,HCES2022655561010131113011 101202 304,1,1,2.0,6.0,2,7.0,8.0,2,30331.0
...,...,...,...,...,...,...,...,...,...,...
261741,HCES20223951022828313220210228101 314,2,1,1.0,6.0,11,8.0,5.0,2,73515.0
261742,HCES20223951022828313220210228101 315,1,1,1.0,6.0,2,5.0,8.0,2,73515.0
261743,HCES20223951022828313220210228101 316,2,1,1.0,6.0,2,5.0,8.0,2,73515.0
261744,HCES20223951022828313220210228101 317,2,1,1.0,6.0,2,8.0,5.0,2,73515.0


In [9]:
df11 = pd.read_fwf('data/raw_hces/hces22_lvl_11.TXT', 
                  header = None, 
                  widths = [38,1,2,1,1,1,1,1,1,1,1,1,1,1,1,3,1,3,1,3,1,3,1,3,1,3,1,3,1,3,1,1,1,1,1,1,1,1,1,1,1,1,1,15], 
                  names=['common_id','q_num','level','clothing','footwear','furniture','recently_bought_mobile_handset',
                        'personal_goods','recreation_goods','cooking','crockery','sports_goods','med_equip','bedding','free_laptop',
                        'num_free_laptop','free_tablet','num_free_tablet','free_mobile','num_free_mobile','free_bicycle',
                        'num_free_bicycle','free_motorcycle','num_free_motorcycle','free_clothing','num_free_clothing',
                        'free_footwear','num_free_footwear','other_free','num_other_free','tv','radio','laptop',
                        'mobile_handset','bicycle','motorcycle','motor_car','trucks','animal_cart','refrigerator',
                        'washing_machine','air_conditioner','multichannel_tv_type','multiplier']
                 )
df11 = df11[['common_id','tv','radio','laptop','mobile_handset','bicycle','motorcycle','motor_car','trucks','animal_cart',
             'refrigerator','washing_machine','air_conditioner','multiplier']]
df11

,common_id,tv,radio,laptop,mobile_handset,bicycle,motorcycle,motor_car,trucks,animal_cart,refrigerator,washing_machine,air_conditioner,multiplier
0,HCES2022655561010131113011 101202 201,1.0,NaN,NaN,1.0,1.0,1.0,1.0,NaN,NaN,1.0,1.0,NaN,35560.0
1,HCES2022655561010131113011 101202 301,1.0,NaN,1.0,1.0,1.0,1.0,NaN,NaN,NaN,1.0,1.0,NaN,30331.0
2,HCES2022655561010131113011 101202 302,1.0,NaN,NaN,1.0,1.0,1.0,NaN,NaN,NaN,1.0,1.0,NaN,30331.0
3,HCES2022655561010131113011 101202 303,1.0,NaN,1.0,1.0,1.0,NaN,1.0,NaN,NaN,1.0,1.0,NaN,30331.0
4,HCES2022655561010131113011 101202 304,1.0,1.0,NaN,1.0,1.0,NaN,NaN,NaN,NaN,1.0,1.0,NaN,30331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261741,HCES20223951022828313220210228101 314,1.0,NaN,NaN,1.0,NaN,1.0,1.0,NaN,NaN,1.0,1.0,NaN,73515.0
261742,HCES20223951022828313220210228101 315,1.0,NaN,NaN,1.0,NaN,1.0,1.0,NaN,NaN,1.0,1.0,1.0,73515.0
261743,HCES20223951022828313220210228101 316,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,1.0,1.0,NaN,73515.0
261744,HCES20223951022828313220210228101 317,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,1.0,1.0,1.0,73515.0


In [10]:
df7 = pd.read_fwf('data/raw_hces/hces22_lvl_07.TXT', 
                  header = None, 
                  widths = [38,1,2,1,1,2,1,1,3,3,1,3,1,3,1,3,1,3,1,2,1,2,1,1,2,8,1,1,1,1,1,1,15], 
                  names=['common_id','q_num','level','kerosene','lpg_subsidy','num_lpg_subsidy','days','attend_education',
                        'unk','num_attend_priv_ed','free_items','free_textbooks','free_stationary','num_free_stationary',
                        'free_school_bag','num_free_school_bag','other_free_items','num_other_free_items','fee_waiver',
                        'num_free_waiver','benefit_pmjay','num_benefit_pmjay','hospitalization','pmjay_hosp_benefit',
                        'pmjay_hosp_benefit_num','pmjay_hosp_benefit_amt','fuel_light','toilet','education','medicine',
                        'services','internet','multiplier']
                 )
df7 = df7[['common_id','internet','multiplier']]
df7

,common_id,internet,multiplier
0,HCES2022655561010131113011 101202 201,2,35560.0
1,HCES2022655561010131113011 101202 301,1,30331.0
2,HCES2022655561010131113011 101202 302,1,30331.0
3,HCES2022655561010131113011 101202 303,1,30331.0
4,HCES2022655561010131113011 101202 304,2,30331.0
...,...,...,...
261741,HCES20223951022828313220210228101 314,1,73515.0
261742,HCES20223951022828313220210228101 315,1,73515.0
261743,HCES20223951022828313220210228101 316,1,73515.0
261744,HCES20223951022828313220210228101 317,1,73515.0


In [11]:
# Merge all HCES wealth dfs together, using 'common_id' as the common key
hces_df = df3.merge(df11, on='common_id')
hces_df = hces_df.merge(df7, on='common_id')
hces_df
# NOTE: With all of these binary variables, 1 means 'Yes' and 2 means 'No' (e.g. if 'washing_machine'==1.0, then yes,
# the household has a washing machine).

,common_id,land_ownership,latrine_access_type,latrine_type,building_material_wall,drinking_water_source,building_material_roof,building_material_floor,energy_cooking,multiplier_x,...,motorcycle,motor_car,trucks,animal_cart,refrigerator,washing_machine,air_conditioner,multiplier_y,internet,multiplier
0,HCES2022655561010131113011 101202 201,1,1,2.0,6.0,2,8.0,8.0,2,35560.0,...,1.0,1.0,NaN,NaN,1.0,1.0,NaN,35560.0,2,35560.0
1,HCES2022655561010131113011 101202 301,1,1,2.0,6.0,2,7.0,8.0,2,30331.0,...,1.0,NaN,NaN,NaN,1.0,1.0,NaN,30331.0,1,30331.0
2,HCES2022655561010131113011 101202 302,1,1,2.0,6.0,2,7.0,8.0,2,30331.0,...,1.0,NaN,NaN,NaN,1.0,1.0,NaN,30331.0,1,30331.0
3,HCES2022655561010131113011 101202 303,1,1,2.0,6.0,2,7.0,8.0,2,30331.0,...,NaN,1.0,NaN,NaN,1.0,1.0,NaN,30331.0,1,30331.0
4,HCES2022655561010131113011 101202 304,1,1,2.0,6.0,2,7.0,8.0,2,30331.0,...,NaN,NaN,NaN,NaN,1.0,1.0,NaN,30331.0,2,30331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261741,HCES20223951022828313220210228101 314,2,1,1.0,6.0,11,8.0,5.0,2,73515.0,...,1.0,1.0,NaN,NaN,1.0,1.0,NaN,73515.0,1,73515.0
261742,HCES20223951022828313220210228101 315,1,1,1.0,6.0,2,5.0,8.0,2,73515.0,...,1.0,1.0,NaN,NaN,1.0,1.0,1.0,73515.0,1,73515.0
261743,HCES20223951022828313220210228101 316,2,1,1.0,6.0,2,5.0,8.0,2,73515.0,...,1.0,NaN,NaN,NaN,1.0,1.0,NaN,73515.0,1,73515.0
261744,HCES20223951022828313220210228101 317,2,1,1.0,6.0,2,8.0,5.0,2,73515.0,...,1.0,NaN,NaN,NaN,1.0,1.0,1.0,73515.0,1,73515.0


In [12]:
# Convert codes to integers so that I can map them to the values in wealth_scores_df
for col in hces_df.select_dtypes(include=['float']).columns:
    # Fill NaN values before conversion to avoid errors
    hces_df[col] = hces_df[col].fillna(0).astype(int)
hces_df

,common_id,land_ownership,latrine_access_type,latrine_type,building_material_wall,drinking_water_source,building_material_roof,building_material_floor,energy_cooking,multiplier_x,...,motorcycle,motor_car,trucks,animal_cart,refrigerator,washing_machine,air_conditioner,multiplier_y,internet,multiplier
0,HCES2022655561010131113011 101202 201,1,1,2,6,2,8,8,2,35560,...,1,1,0,0,1,1,0,35560,2,35560
1,HCES2022655561010131113011 101202 301,1,1,2,6,2,7,8,2,30331,...,1,0,0,0,1,1,0,30331,1,30331
2,HCES2022655561010131113011 101202 302,1,1,2,6,2,7,8,2,30331,...,1,0,0,0,1,1,0,30331,1,30331
3,HCES2022655561010131113011 101202 303,1,1,2,6,2,7,8,2,30331,...,0,1,0,0,1,1,0,30331,1,30331
4,HCES2022655561010131113011 101202 304,1,1,2,6,2,7,8,2,30331,...,0,0,0,0,1,1,0,30331,2,30331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261741,HCES20223951022828313220210228101 314,2,1,1,6,11,8,5,2,73515,...,1,1,0,0,1,1,0,73515,1,73515
261742,HCES20223951022828313220210228101 315,1,1,1,6,2,5,8,2,73515,...,1,1,0,0,1,1,1,73515,1,73515
261743,HCES20223951022828313220210228101 316,2,1,1,6,2,5,8,2,73515,...,1,0,0,0,1,1,0,73515,1,73515
261744,HCES20223951022828313220210228101 317,2,1,1,6,2,8,5,2,73515,...,1,0,0,0,1,1,1,73515,1,73515


In [13]:
columns_to_convert = [
    'latrine_type', 
    'building_material_wall',
    'drinking_water_source',
    'building_material_roof',
    'building_material_floor',
    'energy_cooking'
]

for column in columns_to_convert:
    # Get dummy variables for the current column
    dummies = pd.get_dummies(hces_df[column], prefix=column)
    
    # Change 1s to '1' and 0s to '2'
    dummies = dummies.applymap(lambda x: '1' if x == 1 else '2')
    
    # Drop the original column from hces_df
    hces_df = hces_df.drop(column, axis=1)
    
    # Join the dummy variables to the original DataFrame
    hces_df = pd.concat([hces_df, dummies], axis=1)

In [14]:
for column in hces_df.columns:
    # Check if the column is binary by seeing if it only contains two unique values, 0 and 1
    # (Adjust this logic if needed, for example, to exclude non-binary but two-category columns)
    if set(hces_df[column].unique()).issubset({0, 1}):
        # Convert 0s to 2s (for "no") and keep 1s as is (for "yes")
        hces_df[column] = hces_df[column].map({0: '2', 1: '1'})

In [15]:
hces_df.radio.value_counts()

2    253390
1      8356
Name: radio, dtype: int64

In [16]:
hces_df.columns.tolist()

['common_id',
 'land_ownership',
 'latrine_access_type',
 'multiplier_x',
 'tv',
 'radio',
 'laptop',
 'mobile_handset',
 'bicycle',
 'motorcycle',
 'motor_car',
 'trucks',
 'animal_cart',
 'refrigerator',
 'washing_machine',
 'air_conditioner',
 'multiplier_y',
 'internet',
 'multiplier',
 'latrine_type_0',
 'latrine_type_1',
 'latrine_type_2',
 'latrine_type_3',
 'latrine_type_4',
 'latrine_type_5',
 'latrine_type_6',
 'latrine_type_7',
 'latrine_type_8',
 'latrine_type_10',
 'latrine_type_11',
 'latrine_type_19',
 'building_material_wall_0',
 'building_material_wall_1',
 'building_material_wall_2',
 'building_material_wall_3',
 'building_material_wall_4',
 'building_material_wall_5',
 'building_material_wall_6',
 'building_material_wall_7',
 'building_material_wall_8',
 'building_material_wall_9',
 'drinking_water_source_1',
 'drinking_water_source_2',
 'drinking_water_source_3',
 'drinking_water_source_4',
 'drinking_water_source_5',
 'drinking_water_source_6',
 'drinking_water_sou

In [17]:
hces_df

,common_id,land_ownership,latrine_access_type,multiplier_x,tv,radio,laptop,mobile_handset,bicycle,motorcycle,...,energy_cooking_3,energy_cooking_4,energy_cooking_5,energy_cooking_6,energy_cooking_7,energy_cooking_8,energy_cooking_9,energy_cooking_10,energy_cooking_11,energy_cooking_12
0,HCES2022655561010131113011 101202 201,1,1,35560,1,2,2,1,1,1,...,2,2,2,2,2,2,2,2,2,2
1,HCES2022655561010131113011 101202 301,1,1,30331,1,2,1,1,1,1,...,2,2,2,2,2,2,2,2,2,2
2,HCES2022655561010131113011 101202 302,1,1,30331,1,2,2,1,1,1,...,2,2,2,2,2,2,2,2,2,2
3,HCES2022655561010131113011 101202 303,1,1,30331,1,2,1,1,1,2,...,2,2,2,2,2,2,2,2,2,2
4,HCES2022655561010131113011 101202 304,1,1,30331,1,1,2,1,1,2,...,2,2,2,2,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261741,HCES20223951022828313220210228101 314,2,1,73515,1,2,2,1,2,1,...,2,2,2,2,2,2,2,2,2,2
261742,HCES20223951022828313220210228101 315,1,1,73515,1,2,2,1,2,1,...,2,2,2,2,2,2,2,2,2,2
261743,HCES20223951022828313220210228101 316,2,1,73515,1,2,2,1,2,1,...,2,2,2,2,2,2,2,2,2,2
261744,HCES20223951022828313220210228101 317,2,1,73515,1,2,2,1,2,1,...,2,2,2,2,2,2,2,2,2,2


In [19]:
wealth_scores_dict = wealth_scores_df.set_index('hces_name').to_dict(orient='index')

# Initialize a new column for the wealth score in hces_df
hces_df['wealth_score'] = 0

# Iterate over each row in hces_df to calculate the wealth score
for index, row in hces_df.iterrows():
    # Initialize a temporary score for the current row
    temp_score = 0
    
    # Iterate over each column (variable) in the row
    for column in hces_df.columns:
        # Skip the wealth_score column to avoid trying to look it up
        if column == 'wealth_score':
            continue
        
        # Look up the score for the current variable based on its presence/absence
        score_info = wealth_scores_dict.get(column, None)
        
        if score_info and row[column] == '1':  # If the variable is present
            temp_score += score_info['if_has']
        elif score_info and row[column] == '2':  # If the variable is absent
            temp_score += score_info['if_does_not_have']
    
    # Update the wealth score for the current row in hces_df
    hces_df.at[index, 'wealth_score'] = temp_score

In [20]:
hces_df.wealth_score.describe()

count    261746.000000
mean          0.270701
std           0.502383
min          -1.004478
25%          -0.120821
50%           0.279834
75%           0.648271
max           1.559517
Name: wealth_score, dtype: float64

In [21]:
hces_df['wealth_quintile'] = pd.qcut(hces_df['wealth_score'], 5, labels=['lowest', 'low', 'middle', 'high', 'highest'])
hces_df['wealth_quintile'].value_counts()

high       52367
lowest     52358
low        52350
middle     52340
highest    52331
Name: wealth_quintile, dtype: int64

In [22]:
# Filter hces_df to only have household unique identifier, multiplier (i.e. population weight - will use this later),
# and wealth quintile.
hces_df = hces_df[['common_id', 'multiplier', 'wealth_quintile']]
hces_df

,common_id,multiplier,wealth_quintile
0,HCES2022655561010131113011 101202 201,35560,highest
1,HCES2022655561010131113011 101202 301,30331,highest
2,HCES2022655561010131113011 101202 302,30331,highest
3,HCES2022655561010131113011 101202 303,30331,highest
4,HCES2022655561010131113011 101202 304,30331,high
...,...,...,...
261741,HCES20223951022828313220210228101 314,73515,highest
261742,HCES20223951022828313220210228101 315,73515,highest
261743,HCES20223951022828313220210228101 316,73515,highest
261744,HCES20223951022828313220210228101 317,73515,highest


# Now that I've calculated wealth quintiles, I need to process the rice consumption data and clean up a CSV that I can then tabulate as needed.

## Rice Coverage 
Here we process rice coverage values per household to calculate what percentage of our population of interest consumes ANY rice.

In [23]:
# First load the file that has whether or not a family consumed rice in the last 2 weeks so we can calculate coverage 
df_rice = pd.read_fwf('data/raw_hces/hces22_lvl_04.TXT', 
                 header=None, 
                 widths=[38,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,4,15],
                 names=['common_id','q_num','level','ration_card','rice','wheat','coarse_grain','sugar','pulses',
                        'edible_oil','other_food','groceries','milk','vegetables','fresh_fruit','dry_fruit','animal_prods',
                       'served_proc_food','packed_proc_food','other_food_purchase','ceremony','meals_to_nonhousehold',
                       'multiplier'])
df_rice = df_rice[['common_id','rice']]
df_rice

,common_id,rice
0,HCES2022616751181822223011 118101 311,1.0
1,HCES20226660710808514232010108105 204,NaN
2,HCES2022398952323220812071 132214 101,1.0
3,HCES2022695011202021013018 120153 301,1.0
4,HCES2022346562333343232011 233101 304,NaN
...,...,...
261741,HCES20223951022828313220210228101 313,NaN
261742,HCES20223951022828313220210228101 314,1.0
261743,HCES20223951022828313220210228101 315,NaN
261744,HCES20223951022828313220210228101 316,NaN


In [24]:
df_rice['rice'] = df_rice['rice'].fillna(0).astype('int')
df_rice

,common_id,rice
0,HCES2022616751181822223011 118101 311,1
1,HCES20226660710808514232010108105 204,0
2,HCES2022398952323220812071 132214 101,1
3,HCES2022695011202021013018 120153 301,1
4,HCES2022346562333343232011 233101 304,0
...,...,...
261741,HCES20223951022828313220210228101 313,0
261742,HCES20223951022828313220210228101 314,1
261743,HCES20223951022828313220210228101 315,0
261744,HCES20223951022828313220210228101 316,0


In [25]:
df_rice.rice.value_counts()
# 1 is yes, 0 is no (Whether the household procured rice in the last 30 days)

1    163929
0     97817
Name: rice, dtype: int64

In [26]:
df_rice = df_rice.merge(hces_df, on='common_id')
df_rice

,common_id,rice,multiplier,wealth_quintile
0,HCES2022616751181822223011 118101 311,1,134372,lowest
1,HCES20226660710808514232010108105 204,0,169359,middle
2,HCES2022398952323220812071 132214 101,1,612500,highest
3,HCES2022695011202021013018 120153 301,1,84913,middle
4,HCES2022346562333343232011 233101 304,0,27434,high
...,...,...,...,...
261741,HCES20223951022828313220210228101 313,0,73515,high
261742,HCES20223951022828313220210228101 314,1,73515,highest
261743,HCES20223951022828313220210228101 315,0,73515,highest
261744,HCES20223951022828313220210228101 316,0,73515,highest


In [27]:
# Save as CSV so we can use to tabulate our results across wealth quintiles
df_rice.to_csv('rice_coverage_per_household.csv')

## Rice Consumption Amounts
Here we process the amount of rice each household consumed so that we can calculate the average amount each member of our 
populations of interest (WRA and U5 children) eat per day. 

In [ ]:
df_consumption = pd.read_fwf('data/raw_hces/hces22_lvl_05.TXT', header=None, widths = [38,1,2,3,10,8,10,8,1,15],
                 names=['common_id','q_num','level','item_code','consumption_amt','consumption_value',
                       'total_consumption_amt','total_consumption_value','source','multiplier'])
df_consumption

In [ ]:
nonpds_df = df_consumption[df_consumption['item_code'] == 102]
nonpds_df
# non-PDS rice